# Assignment 4 - Survival Analysis

This notebook walks through the three required survival analysis methods:
1. Kaplan-Meier Analysis
2. Cox Proportional Hazards Model
3. Random Survival Forest

**You can complete the assignment by:**
- Implementing functions in `students/` modules AND running this notebook, OR
- Writing a custom script that calls your implemented functions

**Both approaches are valid** as long as you generate all required outputs.

In [12]:
import os
print(os.getcwd())

/Users/shifa.zafar/Documents/BINF5507/assignment-4-survival-analysis-itsshifazafar-ai/notebooks


In [14]:
%pip install lifelines scikit-survival openpyxl

  Using cached lifelines-0.30.3-py3-none-any.whl.metadata (3.5 kB)
  Using cached autograd_gamma-0.5.0-py3-none-any.whl
  Using cached ecos-2.0.14-cp314-cp314-macosx_10_15_universal2.whl
  Using cached numpy-2.5.1-cp314-cp314-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached markupsafe-3.0.3-cp314-cp314-macosx_11_0_arm64.whl.metadata (2.7 kB)
Using cached lifelines-0.30.3-py3-none-any.whl (409 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 888.8/888.8 kB 27.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 79.5 MB/s  0:00:00
Using cached numpy-2.5.1-cp314-cp314-macosx_14_0_arm64.whl (5.3 MB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
Using cached markupsafe-3.0.3-cp314-cp314-macosx_11_0_arm64.whl (12 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.0 MB/s  0:00:00
  Attempting uninstall: numpy━━━━━━━━━━━━━━━━━━━  0/12 [setuptools]
    Found existing installation: numpy 1.26.4

In [15]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

import sys
import os

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(project_root)

# Import student implementations
from students import (
    fit_kaplan_meier,
    compute_logrank_test,
    plot_km_curves,
    fit_cox_model,
    extract_cox_summary,
    test_proportional_hazards,
    fit_random_survival_forest,
    compute_concordance_index,
    get_feature_importance,
    plot_feature_importance,
    set_plot_style,
)

# Set plotting style
set_plot_style()

# Create outputs directory
Path('outputs').mkdir(exist_ok=True)

/Users/shifa.zafar/Documents/BINF5507/assignment-4-survival-analysis-itsshifazafar-ai


## 1. Load Data

Load the RADCURE clinical dataset.

**Download instructions:**
- Option 1: From TCIA website (see data/README.md)
- Option 2: From Blackboard assignment folder

The file should be saved as `data/RADCURE-clinical-data.xlsx` or `.csv`

In [5]:
# Load RADCURE clinical data
# If you have the Excel file:
data = pd.read_excel('../RADCURE-clinical-data.xlsx')

# If you converted to CSV:
# data = pd.read_csv('../data/RADCURE-clinical-data.csv')

# Display basic info
print(f"Dataset shape: {data.shape}")
print(f"\nFirst few column names: {list(data.columns[:10])}")
print(f"\nData types:\n{data.dtypes.value_counts()}")

# Show first few rows
data.head()

Dataset shape: (3346, 34)

First few column names: ['patient_id', 'Age', 'Sex', 'ECOG PS', 'Smoking PY', 'Smoking Status', 'Ds Site', 'Subsite', 'T', 'N']

Data types:
object            22
datetime64[ns]     7
float64            3
int64              2
Name: count, dtype: int64


,patient_id,Age,Sex,ECOG PS,Smoking PY,Smoking Status,Ds Site,Subsite,T,N,...,Local,Date Local,Regional,Date Regional,Distant,Date Distant,2nd Ca,Date 2nd Ca,RADCURE-challenge,ContrastEnhanced
0,RADCURE-0005,62.6,Female,ECOG 0,50,Ex-smoker,Oropharynx,post wall,T4b,N2c,...,NaN,NaT,NaN,NaT,NaN,NaT,NaN,NaT,0,0
1,RADCURE-0006,87.3,Male,ECOG 2,25,Ex-smoker,Larynx,Glottis,T1b,N0,...,NaN,NaT,NaN,NaT,NaN,NaT,NaN,NaT,0,1
2,RADCURE-0007,49.9,Male,ECOG 1,15,Ex-smoker,Oropharynx,Tonsil,T3,N2b,...,NaN,NaT,NaN,NaT,NaN,NaT,NaN,NaT,0,1
3,RADCURE-0009,72.3,Male,ECOG 1,30,Ex-smoker,Unknown,NaN,T0,N2c,...,NaN,NaT,NaN,NaT,NaN,NaT,S (suspicious),2008-05-27,0,0
4,RADCURE-0010,59.7,Female,ECOG 0,0,Non-smoker,Oropharynx,Tonsillar Fossa,T4b,N0,...,NaN,NaT,NaN,NaT,NaN,NaT,NaN,NaT,0,0


In [7]:
for col in data.columns:
    print(col)

patient_id
Age
Sex
ECOG PS
Smoking PY
Smoking Status
Ds Site
Subsite
T
N
M 
Stage
Path
HPV
Tx Modality
Chemo
RT Start
Dose
Fx
Last FU
Status
Length FU
Date of Death
Cause of Death
Local
Date Local
Regional
Date Regional
Distant
Date Distant
2nd Ca
Date 2nd Ca
RADCURE-challenge
ContrastEnhanced


In [8]:
# Prepare survival variables
# RADCURE column names may vary - check your file!
data["time"] = pd.to_numeric(data["Length FU"], errors="coerce")

data["event"] = (
    data["Status"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "dead": 1,
        "alive": 0
    })
)

time_col = "time"
event_col = "event"

# Verify columns exist
print("Available columns related to survival:")
print([col for col in data.columns if 'time' in col.lower() or 'death' in col.lower() or 'survival' in col.lower()])

# Check event distribution
print(f"\nEvent distribution ({event_col}):")
print(data[event_col].value_counts())
print(f"\nCensoring rate: {(1 - data[event_col].mean())*100:.1f}%")

# Summary statistics
print(f"\nSurvival time (days):")
print(data[time_col].describe())

Available columns related to survival:
['Date of Death', 'Cause of Death', 'time']

Event distribution (event):
event
0    2288
1    1058
Name: count, dtype: int64

Censoring rate: 68.4%

Survival time (days):
count    3346.000000
mean        4.136047
std         2.734757
min         0.156164
25%         1.879452
50%         3.663014
75%         5.791781
max        12.909589
Name: time, dtype: float64


## 2. Kaplan-Meier Analysis

Compare survival curves between patient groups.

In [9]:
# Create grouping variable for KM analysis
# Option 1: Age groups
data['Age_Group'] = pd.cut(data['Age'], bins=[0, 60, 100], labels=['≤60', '>60'])

# Option 2: HPV status (if available)
# group_col = 'HPV_Combined'

# Option 3: Overall stage (Early vs Advanced)
# data['Stage_Group'] = data['Overall_Stage'].map({
#     'I': 'Early', 'II': 'Early',
#     'III': 'Advanced', 'IV': 'Advanced'
# })

# Choose which grouping to use
group_col = 'Age_Group'

print(f"Grouping by: {group_col}")
print(data[group_col].value_counts())

Grouping by: Age_Group
Age_Group
>60    1932
≤60    1414
Name: count, dtype: int64


In [16]:
# Fit KM curves
km_models = fit_kaplan_meier(
    data=data,
    time_col=time_col,
    event_col=event_col,
    group_col=group_col
)

print(f"Fitted KM curves for {len(km_models)} groups: {list(km_models.keys())}")

Fitted KM curves for 2 groups: ['>60', '≤60']


In [17]:
# Compute log-rank test
logrank_results = compute_logrank_test(
    data=data,
    time_col=time_col,
    event_col=event_col,
    group_col=group_col
)

print("\nLog-rank test results:")
print(f"  Test statistic: {logrank_results['test_statistic']:.4f}")
print(f"  p-value: {logrank_results['p_value']:.4f}")

if logrank_results['p_value'] < 0.05:
    print("  → Survival curves are significantly different")
else:
    print("  → No significant difference between groups")


Log-rank test results:
  Test statistic: 136.3387
  p-value: 0.0000
  → Survival curves are significantly different


In [18]:
# Save log-rank results
with open('outputs/logrank_results.json', 'w') as f:
    json.dump(logrank_results, f, indent=2)

print("✅ Saved: outputs/logrank_results.json")

✅ Saved: outputs/logrank_results.json


In [19]:
# Generate KM plot
plot_km_curves(
    km_models=km_models,
    filename='outputs/km_survival_plot.png',
    title=f'Kaplan-Meier Survival Curves by {group_col}'
)

print("✅ Saved: outputs/km_survival_plot.png")

✅ Saved: outputs/km_survival_plot.png


/Users/shifa.zafar/Documents/BINF5507/assignment-4-survival-analysis-itsshifazafar-ai/students/kaplan_meier.py:152: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend(title="Group")


## 3. Cox Proportional Hazards Model

Fit regression model with ≥3 covariates.

In [22]:
# Select covariates (must have at least 3)
# RADCURE example covariates:
covariates = [
    "Age",
    "Sex",
    "Stage",
    "HPV",
    "Tx Modality"
]

print("Cox covariates:", covariates)
print(data[covariates].dtypes)

# Verify columns exist and handle encoding
print("Checking covariates...")
available_covariates = [col for col in covariates if col in data.columns]
print(f"Available: {available_covariates}")

# For categorical variables, may need to encode
# Lifelines handles this automatically, but check data types
for col in available_covariates:
    print(f"{col}: {data[col].dtype}, unique values: {data[col].nunique()}")

# Use at least 3 available covariates
covariates = available_covariates[:6]  # Use first 6 if available
print(f"\nUsing {len(covariates)} covariates: {covariates}")

Cox covariates: ['Age', 'Sex', 'Stage', 'HPV', 'Tx Modality']
Age            float64
Sex             object
Stage           object
HPV             object
Tx Modality     object
dtype: object
Checking covariates...
Available: ['Age', 'Sex', 'Stage', 'HPV', 'Tx Modality']
Age: float64, unique values: 544
Sex: object, unique values: 2
Stage: object, unique values: 14
HPV: object, unique values: 2
Tx Modality: object, unique values: 5

Using 5 covariates: ['Age', 'Sex', 'Stage', 'HPV', 'Tx Modality']


In [23]:
# Fit Cox model
cox = fit_cox_model(
    data=data,
    time_col=time_col,
    event_col=event_col,
    covariates=covariates
)

print("✅ Cox model fitted")

✅ Cox model fitted


In [24]:
# Extract summary
cox_summary = extract_cox_summary(cox)

print("\nCox Model Summary:")
print(cox_summary)


Cox Model Summary:
                      covariate       coef     exp(coef)      se(coef)  \
0                           Age   0.032643  1.033182e+00      0.003178   
1                      Sex_Male   0.081692  1.085122e+00      0.077484   
2                       Stage_I  -0.428026  6.517942e-01      0.261834   
3                      Stage_IB -16.662415  5.802365e-08  17352.719500   
4                      Stage_II  -0.085017  9.184968e-01      0.254365   
5                     Stage_IIA -11.589881  9.259307e-06    605.397672   
6                     Stage_IIB   1.968993  7.163462e+00      1.029319   
7                     Stage_III   0.352237  1.422246e+00      0.247723   
8                    Stage_IIIA   2.608233  1.357504e+01      0.748599   
9                    Stage_IIIC   2.500598  1.218978e+01      0.746966   
10                     Stage_IV   1.005069  2.732095e+00      0.447801   
11                    Stage_IVA   0.936573  2.551222e+00      0.242814   
12                

In [25]:
# Save Cox summary
cox_summary.to_csv('outputs/cox_summary.csv', index=False)
print("✅ Saved: outputs/cox_summary.csv")

✅ Saved: outputs/cox_summary.csv


In [ ]:
# Test proportional hazards assumption
ph_test = test_proportional_hazards(
    cox_model=cox,
    data=data,
    time_col=time_col,
    event_col=event_col
)

print("\nProportional Hazards Test:")
for covar, result in ph_test.items():
    status = "✓" if result['p_value'] > 0.05 else "✗"
    print(f"  {status} {covar}: p = {result['p_value']:.4f}")

NameError: name 'cox_model' is not defined

In [ ]:
# Save PH test results
with open('outputs/cox_ph_test.json', 'w') as f:
    json.dump(ph_test, f, indent=2)

print("✅ Saved: outputs/cox_ph_test.json")

## 4. Random Survival Forest

Train ensemble model and evaluate performance.

In [ ]:
# Prepare features
from sksurv.util import Surv

# Select features (can be same as Cox covariates or different)
feature_cols = covariates
X = data[feature_cols].copy()

# Prepare survival outcome
y = Surv.from_dataframe('event', 'time', data)

print(f"Feature matrix shape: {X.shape}")
print(f"Survival outcome shape: {y.shape}")

In [ ]:
# Train/test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")

In [ ]:
# Fit Random Survival Forest
rsf = fit_random_survival_forest(
    X_train=X_train,
    y_train=y_train,
    n_estimators=100,
    random_state=42
)

print("✅ Random Survival Forest trained")

In [ ]:
# Compute C-index
c_index = compute_concordance_index(rsf, X_test, y_test)

print(f"\nConcordance Index (C-index): {c_index:.4f}")
print("\nInterpretation:")
if c_index > 0.7:
    print("  → Good predictive performance")
elif c_index > 0.6:
    print("  → Moderate predictive performance")
else:
    print("  → Weak predictive performance")

In [ ]:
# Save metrics
rsf_metrics = {'c_index': float(c_index)}

with open('outputs/rsf_metrics.json', 'w') as f:
    json.dump(rsf_metrics, f, indent=2)

print("✅ Saved: outputs/rsf_metrics.json")

In [ ]:
# Get feature importance
importance = get_feature_importance(rsf, feature_cols)

print("\nFeature Importance:")
print(importance)

In [ ]:
# Plot feature importance
plot_feature_importance(
    importance_df=importance,
    filename='outputs/rsf_importance.png',
    top_n=10
)

print("✅ Saved: outputs/rsf_importance.png")

## 5. Verify All Outputs

Check that all required files were generated.

In [ ]:
required_files = [
    'outputs/km_survival_plot.png',
    'outputs/logrank_results.json',
    'outputs/cox_summary.csv',
    'outputs/cox_ph_test.json',
    'outputs/rsf_importance.png',
    'outputs/rsf_metrics.json',
]

print("Checking required outputs:\n")
all_exist = True
for filepath in required_files:
    exists = Path(filepath).exists()
    status = "✅" if exists else "❌"
    print(f"{status} {filepath}")
    all_exist = all_exist and exists

print("\n" + "="*50)
if all_exist:
    print("✅ ALL OUTPUTS GENERATED")
    print("Ready to push to GitHub!")
else:
    print("❌ MISSING OUTPUTS")
    print("Re-run the cells above to generate missing files.")

## Next Steps

1. ✅ Complete `AI_DISCLOSURE.md`
2. ✅ Run `python validate_submission.py` locally
3. ✅ Push to GitHub
4. ✅ Check that GitHub Actions tests pass
5. ✅ Upload outputs to Blackboard (if required)

Good luck! 🎉